# Probabilistic logic programming

[ProbLog](https://dtai.cs.kuleuven.be/problog/) extends Prolog with probabilistic facts and
clauses, written `P::Clause`. A ProbLog solver computes the probability of each solution, i.e. the
probability that the query holds in a world drawn at random from the program.

See also: [Run probabilistic programs](https://tuprolog.github.io/2ppy/how-to/run-probabilistic-programs/).

In [1]:
from tuprolog.solve import solve_options
from tuprolog.solve.plp import probability
from tuprolog.solve.problog import PROBLOG_OPERATORS, problog_solver
from tuprolog.theory.parsing import parse_theory

program = parse_theory(
    """
    0.6::edge(a, b). 0.4::edge(a, c). 0.7::edge(b, d). 0.5::edge(c, d).
    path(X, Y) :- edge(X, Y).
    path(X, Y) :- edge(X, Z), path(Z, Y).
    """,
    PROBLOG_OPERATORS,
)
solver = problog_solver(static_kb=program)

Probabilistic solutions are requested with `solve_options(probabilistic=True)`:

In [2]:
for solution in solver.solve("path(a, d)", solve_options(probabilistic=True)):
    print(solution.solved_query, round(probability(solution), 4))

path(a, d) 0.536


The probability of reaching `d` accounts for both paths (via `b` and via `c`), which are not
mutually exclusive: it is $1 - (1 - 0.6 \cdot 0.7)(1 - 0.4 \cdot 0.5) = 0.536$.

In [3]:
for solution in solver.solve("path(a, N)", solve_options(probabilistic=True)):
    print(solution.substitution, round(probability(solution), 4))

{N_0=b} 0.6
{N_0=c} 0.4
{N_0=d} 0.536
